In [11]:
import numpy as np
import pandas as pd
from mne_bids import BIDSPath, read_raw_bids
from readEEG import HBN_ELECTRODE_MAP
from tqdm import tqdm
import warnings

# Ignore MNE warnings
warnings.filterwarnings("ignore")


In [12]:
def get_dataset_channel_statistics(root):
	"""
	Computes the mean and standard deviation of each channel in the dataset using resting state data.

	Parameters:
		root (str): The root directory of the dataset. Dataset must be in BIDS format.

	Returns:
		statistics : pandas.DataFrame
			A DataFrame containing the mean and standard deviation of each channel in the dataset.
	"""

	bids_path = BIDSPath(root=root, datatype="eeg", suffix="eeg", extension=".set")

	subjects = sorted(set([bp.subject for bp in tqdm(bids_path.match())]))

	print(f"Found {len(subjects)} subjects")

	electrodes = [f'E{i}' for i in range(1, 129)]
	to_drop = [electrode for electrode in electrodes if electrode not in HBN_ELECTRODE_MAP.values()]


	indices = None
	means = pd.DataFrame()
	stds = pd.DataFrame()

	HBN_ELECTRODE_MAP_REVERSED = dict((v,k) for k,v in HBN_ELECTRODE_MAP.items())

	for subject in subjects:
		print(f"Processing subject: {subject}")
		subject_all_data = []
		bids_path.update(subject=subject)
		for bp in tqdm(bids_path.match()):
			raw = read_raw_bids(bp, verbose=False)
			raw.drop_channels(to_drop)
			raw.rename_channels(HBN_ELECTRODE_MAP_REVERSED)
			if indices is None:
				indices = raw.ch_names
			subject_all_data.append(raw.get_data())

		subject_all_data = np.concatenate(subject_all_data, axis=1)

		# Z-score Transform By Channel
		mean = np.mean(subject_all_data, axis=1)
		std = np.std(subject_all_data, axis=1)

		means[subject] = mean
		stds[subject] = std
		break
	
	means.set_index(pd.Index(indices), inplace=True)
	stds.set_index(pd.Index(indices), inplace=True)
	print(means.head())
	print(stds.head())


In [13]:
get_dataset_channel_statistics("~/mycontainer/eegdata/ds005505-download/")

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1342/1342 [00:00<00:00, 2602291.25it/s]


Found 136 subjects
Processing subject: NDARAC904DMU


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [00:30<00:00,  2.53s/it]

     NDARAC904DMU
Fp2      0.057892
Fz       0.069840
Fp1      0.054869
F3       0.061454
F7       0.022064
     NDARAC904DMU
Fp2      0.019575
Fz       0.020135
Fp1      0.017864
F3       0.019160
F7       0.009907
